In [0]:
silver_schema_name = 'ws_azuredatabricks.silver'
silver_table_name = 'hospital'

hospital_df = spark.table(f'{silver_schema_name}.{silver_table_name}')
display(hospital_df)

hospital_id,hospital_name,city,bed_count
H002,Fortis Healthcare Delhi,Delhi,600
H003,Kokilaben Dhirubhai Hospital,Mumbai,700
H001,Apollo Main Hospital,Chennai,850
H005,Yashoda Hospital,Hyderabad,450
H004,Manipal Hospital,Bangalore,400


In [0]:
%sql

CREATE TABLE IF NOT EXISTS ws_azuredatabricks.gold.dim_hospital(
    hospital_id string NOT NULL,
    hospital_name string NOT NULL,
    city string NOT NULL,
    bed_count integer NOT NULL,
    row_hash string NOT NULL,
    created_at timestamp NOT NULL,
    updated_at timestamp NOT NULL
)


In [0]:
%sql
CREATE VIEW IF NOT EXISTS ws_azuredatabricks.gold.hospital_source AS
SELECT 
hospital_id,
hospital_name,
city,
bed_count,
sha2(
    concat_ws('||',
    coalesce(hospital_name,'<null>'),
    coalesce(city,'<null>'),
    coalesce(cast(bed_count as string), '<null>')), 256) as row_hash
FROM ws_azuredatabricks.silver.hospital

In [0]:
%sql
SELECT * FROM ws_azuredatabricks.gold.hospital_source

hospital_id,hospital_name,city,bed_count,row_hash
H002,Fortis Healthcare Delhi,Delhi,600,2f28060427b7938a099a62436b782be781fb4f2bcbabbd2350f2ac2319076ea5
H003,Kokilaben Dhirubhai Hospital,Mumbai,700,7a90cf2d949d851850f7887e0a3258afd5124b2755e64ad582c97d8dbc6fd9a2
H001,Apollo Main Hospital,Chennai,850,3b7e70fefaa4f1c44d3f689d698a5d00a97313d8d67b7bcd73bcba4358e66a1c
H005,Yashoda Hospital,Hyderabad,450,39fbd8775b54e38e7b1b937241e7c186ed19edb965a44082d101541d06ae1884
H004,Manipal Hospital,Bangalore,400,148fb2754618722bd51bdb8bcbe0dadb6bbf2a466bf9429706bf32f6014be544


In [0]:
%sql
MERGE INTO ws_azuredatabricks.gold.dim_hospital as TARGET
USING ws_azuredatabricks.gold.hospital_source as SOURCE
ON TARGET.hospital_id = SOURCE.hospital_id
WHEN MATCHED AND TARGET.row_hash <> SOURCE.row_hash THEN UPDATE SET
TARGET.hospital_name = SOURCE.hospital_name,
TARGET.city = SOURCE.city,
TARGET.bed_count = SOURCE.bed_count,
TARGET.row_hash = SOURCE.row_hash,
TARGET.updated_at = current_timestamp()
WHEN NOT MATCHED THEN INSERT 
(hospital_id, hospital_name, city, bed_count, row_hash, created_at, updated_at)
VALUES 
(SOURCE.hospital_id, SOURCE.hospital_name, SOURCE.city, SOURCE.bed_count, SOURCE.row_hash, current_timestamp(), current_timestamp())

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5,0,0,5


In [0]:
%sql
SELECT * FROM ws_azuredatabricks.gold.dim_hospital

hospital_id,hospital_name,city,bed_count,row_hash,created_at,updated_at
H002,Fortis Healthcare Delhi,Delhi,600,2f28060427b7938a099a62436b782be781fb4f2bcbabbd2350f2ac2319076ea5,2026-11-01T00:34:15.775Z,2026-11-01T00:34:15.775Z
H003,Kokilaben Dhirubhai Hospital,Mumbai,700,7a90cf2d949d851850f7887e0a3258afd5124b2755e64ad582c97d8dbc6fd9a2,2026-11-01T00:34:15.775Z,2026-11-01T00:34:15.775Z
H001,Apollo Main Hospital,Chennai,850,3b7e70fefaa4f1c44d3f689d698a5d00a97313d8d67b7bcd73bcba4358e66a1c,2026-11-01T00:34:15.775Z,2026-11-01T00:34:15.775Z
H005,Yashoda Hospital,Hyderabad,450,39fbd8775b54e38e7b1b937241e7c186ed19edb965a44082d101541d06ae1884,2026-11-01T00:34:15.775Z,2026-11-01T00:34:15.775Z
H004,Manipal Hospital,Bangalore,400,148fb2754618722bd51bdb8bcbe0dadb6bbf2a466bf9429706bf32f6014be544,2026-11-01T00:34:15.775Z,2026-11-01T00:34:15.775Z


Test SCD Type 1

In [0]:
%sql
CREATE VIEW IF NOT EXISTS ws_azuredatabricks.gold.hospital_test as
SELECT 
hospital_id,
hospital_name,
city,
CASE WHEN hospital_id = 'H002' THEN 1000 ELSE bed_count END as bed_count
FROM ws_azuredatabricks.silver.hospital

In [0]:
%sql
SELECT * FROM ws_azuredatabricks.gold.hospital_test

hospital_id,hospital_name,city,bed_count
H002,Fortis Healthcare Delhi,Delhi,1000
H003,Kokilaben Dhirubhai Hospital,Mumbai,700
H001,Apollo Main Hospital,Chennai,850
H005,Yashoda Hospital,Hyderabad,450
H004,Manipal Hospital,Bangalore,400


In [0]:
%sql
CREATE OR REPLACE VIEW ws_azuredatabricks.gold.hospital_source AS
SELECT 
hospital_id,
hospital_name,
city,
bed_count,
sha2(
    concat_ws('||',
    coalesce(hospital_name,'<null>'),
    coalesce(city,'<null>'),
    coalesce(cast(bed_count as string), '<null>')), 256) as row_hash
FROM ws_azuredatabricks.gold.hospital_test

In [0]:
%sql
SELECT * FROM ws_azuredatabricks.gold.hospital_source

hospital_id,hospital_name,city,bed_count,row_hash
H002,Fortis Healthcare Delhi,Delhi,1000,573bebeefa3eff116ad9317e03e99884ceff4674ac6bb411c9b0b7e36c061bbc
H003,Kokilaben Dhirubhai Hospital,Mumbai,700,7a90cf2d949d851850f7887e0a3258afd5124b2755e64ad582c97d8dbc6fd9a2
H001,Apollo Main Hospital,Chennai,850,3b7e70fefaa4f1c44d3f689d698a5d00a97313d8d67b7bcd73bcba4358e66a1c
H005,Yashoda Hospital,Hyderabad,450,39fbd8775b54e38e7b1b937241e7c186ed19edb965a44082d101541d06ae1884
H004,Manipal Hospital,Bangalore,400,148fb2754618722bd51bdb8bcbe0dadb6bbf2a466bf9429706bf32f6014be544


In [0]:
%sql
MERGE INTO ws_azuredatabricks.gold.dim_hospital as TARGET
USING ws_azuredatabricks.gold.hospital_source as SOURCE
ON TARGET.hospital_id = SOURCE.hospital_id
WHEN MATCHED AND TARGET.row_hash <> SOURCE.row_hash THEN UPDATE SET
TARGET.hospital_name = SOURCE.hospital_name,
TARGET.city = SOURCE.city,
TARGET.bed_count = SOURCE.bed_count,
TARGET.row_hash = SOURCE.row_hash,
TARGET.updated_at = current_timestamp()
WHEN NOT MATCHED THEN INSERT 
(hospital_id, hospital_name, city, bed_count, row_hash, created_at, updated_at)
VALUES 
(SOURCE.hospital_id, SOURCE.hospital_name, SOURCE.city, SOURCE.bed_count, SOURCE.row_hash, current_timestamp(), current_timestamp())

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
1,1,0,0


In [0]:
%sql
SELECT * from ws_azuredatabricks.gold.dim_hospital ORDER BY hospital_id

hospital_id,hospital_name,city,bed_count,row_hash,created_at,updated_at
H001,Apollo Main Hospital,Chennai,850,3b7e70fefaa4f1c44d3f689d698a5d00a97313d8d67b7bcd73bcba4358e66a1c,2026-11-01T00:34:15.775Z,2026-11-01T00:34:15.775Z
H002,Fortis Healthcare Delhi,Delhi,1000,573bebeefa3eff116ad9317e03e99884ceff4674ac6bb411c9b0b7e36c061bbc,2026-11-01T00:34:15.775Z,2026-11-01T00:44:08.437Z
H003,Kokilaben Dhirubhai Hospital,Mumbai,700,7a90cf2d949d851850f7887e0a3258afd5124b2755e64ad582c97d8dbc6fd9a2,2026-11-01T00:34:15.775Z,2026-11-01T00:34:15.775Z
H004,Manipal Hospital,Bangalore,400,148fb2754618722bd51bdb8bcbe0dadb6bbf2a466bf9429706bf32f6014be544,2026-11-01T00:34:15.775Z,2026-11-01T00:34:15.775Z
H005,Yashoda Hospital,Hyderabad,450,39fbd8775b54e38e7b1b937241e7c186ed19edb965a44082d101541d06ae1884,2026-11-01T00:34:15.775Z,2026-11-01T00:34:15.775Z


SCD Type 2 for Patient table 

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ws_azuredatabricks.gold.dim_patient(
    patient_id string NOT NULL,
    first_name string NOT NULL,
    last_name string NOT NULL,
    gender string NOT NULL,
    dob date NOT NULL,
    city string NOT NULL,
    row_hash string NOT NULL,
    created_at timestamp NOT NULL,
    updated_at timestamp NOT NULL,
    is_current BOOLEAN NOT NULL
)

In [0]:
%sql
CREATE VIEW IF NOT EXISTS ws_azuredatabricks.gold.patient_source AS
SELECT
patient_id,
first_name,
last_name,
gender,
dob,
city,
sha2(
    concat_ws('||',
    coalesce(first_name,'<null>'),
    coalesce(last_name,'<null>'),
    coalesce(gender, '<null>'),
    coalesce(dob, '<null>'),
    coalesce(city, '<null>')), 256) as row_hash
FROM ws_azuredatabricks.silver.patient


In [0]:
%sql
SELECT * FROM ws_azuredatabricks.gold.patient_source ORDER BY patient_id

patient_id,first_name,last_name,gender,dob,city,row_hash
P001,Aarav,Sharma,M,1980-02-14,Mumbai,e66c833e6480d00be5a356a4d4cde485b1715f2225101f0a46c104ab7e38f6c1
P002,Riya,Verma,F,1975-07-22,Delhi,ca1b01087eacc656705e200f3a52c337c0f058e2f4afc774cba1d95c5a2e4291
P003,Kabir,Menon,M,1990-11-02,Bangalore,341355821870720fff4583b35a0ddd2a54196228496efefa4bb1802ae11c52a6
P004,Sneha,Rao,F,1988-09-19,Hyderabad,ce092fa83367259e089522ac5f9bc9a60a8586b7fcc296984677d508d15d51a3
P005,Vikram,Singh,M,1965-05-05,Chennai,4c531589d1b33add3259279225055f9918bdda87c6ae070daee11647240a027f


In [0]:
%sql
MERGE INTO ws_azuredatabricks.gold.dim_patient AS TARGET
USING ws_azuredatabricks.gold.patient_source AS SOURCE
ON TARGET.patient_id = SOURCE.patient_id AND TARGET.is_current = true

WHEN MATCHED AND TARGET.row_hash <> SOURCE.row_hash THEN UPDATE SET
TARGET.is_current = false,
TARGET.updated_at = current_timestamp()


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
%sql
INSERT INTO ws_azuredatabricks.gold.dim_patient
  (patient_id, first_name, last_name, gender, dob, city, row_hash,
   created_at, updated_at, is_current)
SELECT
  SOURCE.patient_id, SOURCE.first_name, SOURCE.last_name, SOURCE.gender, SOURCE.dob, SOURCE.city, SOURCE.row_hash, current_timestamp(), current_timestamp(), true                                     
FROM ws_azuredatabricks.gold.patient_source AS SOURCE
LEFT ANTI JOIN (
  SELECT patient_id FROM ws_azuredatabricks.gold.dim_patient WHERE is_current = true
) AS current_rows
ON SOURCE.patient_id = current_rows.patient_id;

num_affected_rows,num_inserted_rows
5,5


In [0]:
%sql
SELECT * FROM ws_azuredatabricks.gold.dim_patient ORDER BY patient_id

patient_id,first_name,last_name,gender,dob,city,row_hash,created_at,updated_at,is_current
P001,Aarav,Sharma,M,1980-02-14,Mumbai,e66c833e6480d00be5a356a4d4cde485b1715f2225101f0a46c104ab7e38f6c1,2026-11-01T01:27:18.289Z,2026-11-01T01:27:18.289Z,true
P002,Riya,Verma,F,1975-07-22,Delhi,ca1b01087eacc656705e200f3a52c337c0f058e2f4afc774cba1d95c5a2e4291,2026-11-01T01:27:18.289Z,2026-11-01T01:27:18.289Z,true
P003,Kabir,Menon,M,1990-11-02,Bangalore,341355821870720fff4583b35a0ddd2a54196228496efefa4bb1802ae11c52a6,2026-11-01T01:27:18.289Z,2026-11-01T01:27:18.289Z,true
P004,Sneha,Rao,F,1988-09-19,Hyderabad,ce092fa83367259e089522ac5f9bc9a60a8586b7fcc296984677d508d15d51a3,2026-11-01T01:27:18.289Z,2026-11-01T01:27:18.289Z,true
P005,Vikram,Singh,M,1965-05-05,Chennai,4c531589d1b33add3259279225055f9918bdda87c6ae070daee11647240a027f,2026-11-01T01:27:18.289Z,2026-11-01T01:27:18.289Z,true


Test SCD Type 2 for patient Table 

In [0]:
%sql
CREATE VIEW IF NOT EXISTS ws_azuredatabricks.gold.patient_test AS
SELECT 
patient_id,
first_name,
last_name,
gender,
dob,
CASE WHEN patient_id = 'P001' THEN 'Noida' ELSE city END as city
FROM ws_azuredatabricks.silver.patient
UNION ALL 
SELECT 'P006', 'Abhishek', 'Gambhir', 'M', '1996-02-18','Delhi' 

In [0]:
%sql
SELECT * from ws_azuredatabricks.gold.patient_test ORDER BY patient_id

patient_id,first_name,last_name,gender,dob,city
P001,Aarav,Sharma,M,1980-02-14,Noida
P002,Riya,Verma,F,1975-07-22,Delhi
P003,Kabir,Menon,M,1990-11-02,Bangalore
P004,Sneha,Rao,F,1988-09-19,Hyderabad
P005,Vikram,Singh,M,1965-05-05,Chennai
P006,Abhishek,Gambhir,M,1996-02-18,Delhi


In [0]:
%sql
CREATE OR REPLACE VIEW ws_azuredatabricks.gold.patient_source AS
SELECT
patient_id,
first_name,
last_name,
gender,
dob,
city,
sha2(
    concat_ws('||',
    coalesce(first_name,'<null>'),
    coalesce(last_name,'<null>'),
    coalesce(gender, '<null>'),
    coalesce(dob, '<null>'),
    coalesce(city, '<null>')), 256) as row_hash
FROM ws_azuredatabricks.gold.patient_test

In [0]:
%sql
SELECT * from ws_azuredatabricks.gold.patient_source ORDER BY patient_id

patient_id,first_name,last_name,gender,dob,city,row_hash
P001,Aarav,Sharma,M,1980-02-14,Noida,1c437ef31409853a1ddd064db2cd411f24e3a90af307958bffb41e76d4e6606f
P002,Riya,Verma,F,1975-07-22,Delhi,ca1b01087eacc656705e200f3a52c337c0f058e2f4afc774cba1d95c5a2e4291
P003,Kabir,Menon,M,1990-11-02,Bangalore,341355821870720fff4583b35a0ddd2a54196228496efefa4bb1802ae11c52a6
P004,Sneha,Rao,F,1988-09-19,Hyderabad,ce092fa83367259e089522ac5f9bc9a60a8586b7fcc296984677d508d15d51a3
P005,Vikram,Singh,M,1965-05-05,Chennai,4c531589d1b33add3259279225055f9918bdda87c6ae070daee11647240a027f
P006,Abhishek,Gambhir,M,1996-02-18,Delhi,f200d61f8429b36d7fb7d3e61c9d0c54cd9d8ee008e49c894bd088a676b8384f


In [0]:
%sql
SELECT * from ws_azuredatabricks.gold.dim_patient ORDER BY patient_id

patient_id,first_name,last_name,gender,dob,city,row_hash,created_at,updated_at,is_current
P001,Aarav,Sharma,M,1980-02-14,Mumbai,e66c833e6480d00be5a356a4d4cde485b1715f2225101f0a46c104ab7e38f6c1,2026-11-01T01:27:18.289Z,2026-11-01T01:27:18.289Z,true
P002,Riya,Verma,F,1975-07-22,Delhi,ca1b01087eacc656705e200f3a52c337c0f058e2f4afc774cba1d95c5a2e4291,2026-11-01T01:27:18.289Z,2026-11-01T01:27:18.289Z,true
P003,Kabir,Menon,M,1990-11-02,Bangalore,341355821870720fff4583b35a0ddd2a54196228496efefa4bb1802ae11c52a6,2026-11-01T01:27:18.289Z,2026-11-01T01:27:18.289Z,true
P004,Sneha,Rao,F,1988-09-19,Hyderabad,ce092fa83367259e089522ac5f9bc9a60a8586b7fcc296984677d508d15d51a3,2026-11-01T01:27:18.289Z,2026-11-01T01:27:18.289Z,true
P005,Vikram,Singh,M,1965-05-05,Chennai,4c531589d1b33add3259279225055f9918bdda87c6ae070daee11647240a027f,2026-11-01T01:27:18.289Z,2026-11-01T01:27:18.289Z,true


In [0]:
%sql
MERGE INTO ws_azuredatabricks.gold.dim_patient AS TARGET
USING ws_azuredatabricks.gold.patient_source AS SOURCE
ON TARGET.patient_id = SOURCE.patient_id AND TARGET.is_current = true
WHEN MATCHED AND TARGET.row_hash <> SOURCE.row_hash THEN UPDATE SET
TARGET.is_current = false,
TARGET.updated_at = current_timestamp()

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
%sql
SELECT * from ws_azuredatabricks.gold.dim_patient

patient_id,first_name,last_name,gender,dob,city,row_hash,created_at,updated_at,is_current
P005,Vikram,Singh,M,1965-05-05,Chennai,4c531589d1b33add3259279225055f9918bdda87c6ae070daee11647240a027f,2026-11-01T01:27:18.289Z,2026-11-01T01:27:18.289Z,true
P003,Kabir,Menon,M,1990-11-02,Bangalore,341355821870720fff4583b35a0ddd2a54196228496efefa4bb1802ae11c52a6,2026-11-01T01:27:18.289Z,2026-11-01T01:27:18.289Z,true
P002,Riya,Verma,F,1975-07-22,Delhi,ca1b01087eacc656705e200f3a52c337c0f058e2f4afc774cba1d95c5a2e4291,2026-11-01T01:27:18.289Z,2026-11-01T01:27:18.289Z,true
P004,Sneha,Rao,F,1988-09-19,Hyderabad,ce092fa83367259e089522ac5f9bc9a60a8586b7fcc296984677d508d15d51a3,2026-11-01T01:27:18.289Z,2026-11-01T01:27:18.289Z,true
P001,Aarav,Sharma,M,1980-02-14,Mumbai,e66c833e6480d00be5a356a4d4cde485b1715f2225101f0a46c104ab7e38f6c1,2026-11-01T01:27:18.289Z,2026-11-01T02:16:00.143Z,false


In [0]:
%sql
INSERT INTO ws_azuredatabricks.gold.dim_patient
  (patient_id, first_name, last_name, gender, dob, city, row_hash,
   created_at, updated_at, is_current)
SELECT
  SOURCE.patient_id, SOURCE.first_name, SOURCE.last_name, SOURCE.gender, SOURCE.dob, SOURCE.city, SOURCE.row_hash, current_timestamp(), current_timestamp(), true                                     
FROM ws_azuredatabricks.gold.patient_source AS SOURCE
LEFT ANTI JOIN (
  SELECT patient_id FROM ws_azuredatabricks.gold.dim_patient WHERE is_current = true
) AS current_rows
ON SOURCE.patient_id = current_rows.patient_id;

num_affected_rows,num_inserted_rows
2,2


In [0]:
%sql
SELECT * from ws_azuredatabricks.gold.dim_patient ORDER BY patient_id

patient_id,first_name,last_name,gender,dob,city,row_hash,created_at,updated_at,is_current
P001,Aarav,Sharma,M,1980-02-14,Noida,1c437ef31409853a1ddd064db2cd411f24e3a90af307958bffb41e76d4e6606f,2026-11-01T02:18:35.605Z,2026-11-01T02:18:35.605Z,true
P001,Aarav,Sharma,M,1980-02-14,Mumbai,e66c833e6480d00be5a356a4d4cde485b1715f2225101f0a46c104ab7e38f6c1,2026-11-01T01:27:18.289Z,2026-11-01T02:16:00.143Z,false
P002,Riya,Verma,F,1975-07-22,Delhi,ca1b01087eacc656705e200f3a52c337c0f058e2f4afc774cba1d95c5a2e4291,2026-11-01T01:27:18.289Z,2026-11-01T01:27:18.289Z,true
P003,Kabir,Menon,M,1990-11-02,Bangalore,341355821870720fff4583b35a0ddd2a54196228496efefa4bb1802ae11c52a6,2026-11-01T01:27:18.289Z,2026-11-01T01:27:18.289Z,true
P004,Sneha,Rao,F,1988-09-19,Hyderabad,ce092fa83367259e089522ac5f9bc9a60a8586b7fcc296984677d508d15d51a3,2026-11-01T01:27:18.289Z,2026-11-01T01:27:18.289Z,true
P005,Vikram,Singh,M,1965-05-05,Chennai,4c531589d1b33add3259279225055f9918bdda87c6ae070daee11647240a027f,2026-11-01T01:27:18.289Z,2026-11-01T01:27:18.289Z,true
P006,Abhishek,Gambhir,M,1996-02-18,Delhi,f200d61f8429b36d7fb7d3e61c9d0c54cd9d8ee008e49c894bd088a676b8384f,2026-11-01T02:18:35.605Z,2026-11-01T02:18:35.605Z,true
